## ImmunoGeNN

![ImmunoGeNN Logo](../img/logo_075.png)

ImmunoGeNN accepts input protein sequences and predicts population-level immunogenicty risk scores (MHC-II) based on allele frequencies in the global population. It also supports deimmunization and immunization of the first input sequence by screening all single amino acid variants (SAVs) for predicted immunogenicity risk changes and sequence likelihood (ESM2). Read more in our [EurIPS 2025 SIMBIOCHEM workshop paper](https://openreview.net/forum?id=kOJQm9YXnB).

Example FASTA input:
```
>LYZL4_MOUSE Lysozyme-like protein 4
MQLYLVLLLISYLLTPIGASILGRCTVAKMLYDGGLNYFEGYSLENWVCLAYFESKFNPS
AVYEDPQDGSTGFGLFQIRDNEWCGHGKNLCSVSCTALLNPNLKDTIQCAKKIVKGKHGM
GAWPIWSKNCQLSDVLDRWLDGCDL
```

__Web-servers and code:__
- [Biolib web-server](https://biolib.com/DTU/ImmunoGeNN)
- [DTU Health web-server](https://services.healthtech.dtu.dk/services/ImmunoGeNN/)
- [GitHub repository](https://github.com/novonordisk-research/ImmunoGeNN)

## Quick start (local installation with pip)

In [ ]:
pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ immunogenn --upgrade --no-cache-dir

##### Confirm installation:

In [ ]:
import sys
import immunogenn

print(f"Python: {sys.executable}")
print(f"Module location: {immunogenn.__file__}")
print(immunogenn.__version__)

print("✓ All imports successful!")

#### Simple run:

In [ ]:
import immunogenn

# Screen protein sequences for predicted immunogenicity risk in global population (pIRS)
immunogenn.predict_immunogenicity(
    fasta_file="./examples/input.fasta"
)

## Argument Reference

### Common args
| Argument | Type | Default | Description |
|----------|------|---------|-------------|
| `fasta_file` | `str` | `data/input.fasta` | Path to input FASTA file with protein sequences (min 15 residues) |
| `mode` | `str` | `screen` | Prediction mode: `screen`, `deimmunize`, or `immunize` |
| `outdir` | `str` | `output` | Output directory for results and plots |

### Advanced args
| Argument | Type | Default | Description |
|----------|------|---------|-------------|
| `human_references_pkl` | `bool` | `true` | Flag binding cores in the human proteome (including VJ germlines) |
| `extra_references` | `str` | `""` | Optional path to additional reference sequences in FASTA format |
| `variants_to_generate` | `int` | `30` | Number of de/immunizing variants to generate (deimmunize/immunize modes only) |
| `ranges_str` | `str` | `""` | Optional position ranges to de/immunize (e.g., `30-45,90-120`). Empty = all positions |

**Key notes:**
- `fasta_file` defaults to the bundled example FASTA; provide your own path to screen your sequences
- `mode=screen` (default) predicts immunogenicity for all sequences
- `mode=deimmunize` or `mode=immunize` optimizes the first sequence by screening all single amino acid variants
- `human_references_pkl` enabled by default; set to `false` to disable human proteome screening
- Variant filtering uses ESM2 and pIRS ranking thresholds (hardcoded defaults: ESM ≥60%, pIRS ≥80%)

### Deimmunize first sequence

In [ ]:
import immunogenn

immunogenn.predict_immunogenicity(
    fasta_file="./examples/input.fasta",
    mode="deimmunize",
    outdir="./deimmunize_output"
)

### Immunize first sequence

In [ ]:
import immunogenn

immunogenn.predict_immunogenicity(
    fasta_file="./examples/input.fasta",
    mode="immunize",
    outdir="./immunize_output"
)

### Output format

ImmunoGeNN predicts per-peptide IRS scores ("pIRS") for the Global population, DRB1 gene class. Sequence pIRS_sum scores are calculated by summing across all peptide pIRS scores in the given sequence.

**pIRS score interpretation:** We suggest using a pIRS rank threshold of ~83% to identify immunogenic peptides, as described in the paper. Higher scores indicate higher global population (MHC-II presentation) immunogenicity risk, with an estimated experimental Spearman R MAPPs correlation of ~0.45.

### Example output

pIRS.csv - CSV file containing per-peptide pIRS scores
```csv
id,peptide_pos,gene_class,peptide_seq,core_pos,core_seq,pIRS,pIRS_rank,in_reference,core_0,core_1,core_2,core_3,core_4,core_5,core_6
design1,1,DRB1,EVQLLESGGEVKKPG,3,LLESGGEVK,0.03498,44.949,,0.00,0.00,13.96,67.79,18.25,0.00,0.00
design1,2,DRB1,VQLLESGGEVKKPGA,3,LESGGEVKK,0.03122,36.630,,11.49,0.00,31.90,56.62,0.00,0.00,0.00
design1,3,DRB1,QLLESGGEVKKPGAS,2,LESGGEVKK,0.02773,20.586,,0.00,14.51,71.27,0.00,0.00,0.00,14.22
```
- id: Sequence identifier from input FASTA
- peptide_pos: Position of peptide in sequence (1-indexed)
- gene_class: Always MHC-II DRB1 gene class
- peptide_seq: Peptide sequence
- core_pos: Position of dominant DRB1 binding core in peptide (0-indexed)
- core_seq: DRB1 binding core sequence
- pIRS: Predicted immunogenicity risk score. Higher is more immunogenic
- pIRS_rank: Predicted rank in NetMHCIIpan-4.3 training set (see paper). Higher is more immunogenic above a threshold of ~83%.
- in_reference: Set to True if the 9-mer core sequence is found in human reference proteome
- core_0 to core_6: Predicted peptide IRS for all 7 binding cores, corresponding to model confidence. The top binding core is picked based on the highest pIRS score. See paper for more details.

scores.csv - CSV file containing per-sequence pIRS scores (summed across all peptides)
```csv
id,population,DRB1_pIRS_sum
design1,Global,5.16455
design2,Global,5.17534
design3,Global,5.13089
```
- id: Sequence identifier from input FASTA
- population: Population for which pIRS is calculated
- DRB1_pIRS_sum: Sum of pIRS scores across all peptides in the sequence

### Deimmunization / Immunization visualization
Visualizes the immunogenicity effect of peptide variants across the sequence:
- ![Deimmunization](../img/deimmunization_example.png)
- ![Immunogenicity heatmap](../img/heatmap.png)

----

### Citation
```.bib
@inproceedings{
    hoie2025_immunogenn,
    title={ImmunoGe{NN}: Accelerating Early Immunogenicity Assessment for Generative Design of Biologics},
    author={Magnus Haraldson H{\o}ie and Birkir Reynisson and Paolo Marcatili and Jesper Ferkinghoff-Borg and Kasper Lamberth and Katharina L. Kopp and Morten Nielsen and Vanessa Isabell Jurtz},
    booktitle={EurIPS 2025 Workshop on SIMBIOCHEM},
    year={2025},
    url={https://openreview.net/forum?id=kOJQm9YXnB}
}
```

### License
This project is licensed under the MIT License - see the LICENSE file for details.